# Change Data Feed (CDF) med Delta Sharing

Dette notebooken demonstrerer hvordan du bruker **Change Data Feed (CDF)** til å spore endringer i Delta-tabeller over tid.

In [11]:
! pip install -r requirements.txt
! -packages io.delta:delta-sharing-spark_2.12:3.1.0


zsh:1: command not found: -packages


In [12]:
import os
import json
import delta_sharing
import pandas as pd
from google.cloud import storage
from datetime import datetime, timedelta
import utils.change_data_feed_utils as cdf_utils



Vi henter config.share ved å kjøre "skyporten-deltashare"-repoet 

In [13]:
# Spesifiser sti til din Delta Sharing config-fil
# Denne filen inneholder credentials og endpoint for din share som du henter ved å kjøre 
CONFIG_FILE = "share/config.share"  # Endre til din config-fil

In [14]:
# Koble til Delta Sharing
sharing_client = delta_sharing.SharingClient(CONFIG_FILE)
tables = sharing_client.list_all_tables()

if not tables:
    raise Exception("❌ Ingen tabeller funnet i sharen")

# Filtrer bort tabeller som ikke er gode for demonstrasjon
# (kode-tabeller, nøkkel-tabeller, krypterte tabeller)
not_valid_table_names = ["kode", "keys", "encrypted"]
valid_examples_tables = [
    table for table in tables
    if not any(substr in table.name for substr in not_valid_table_names)
]

# Velg første egnede tabell (eller første tabell hvis ingen egnede finnes)
table = valid_examples_tables[0] if len(valid_examples_tables) > 0 else tables[0]

# Bygg full tabell-URL for Delta Sharing
table_url = f"{CONFIG_FILE}#{table.share}.{table.schema}.{table.name}"

print(f"✓ Valgt tabell: {table.name}")
print(f"  Share: {table.share}")
print(f"  Schema: {table.schema}")
print(f"  Full URL: {table_url}")

✓ Valgt tabell: dim_kulturminner
  Share: 9ivj-dev
  Schema: matrikkel_silver_v1_ext
  Full URL: share/config.share#9ivj-dev.matrikkel_silver_v1_ext.dim_kulturminner


##  Opprett Spark Session



In [15]:
from pyspark.sql import SparkSession

def ensure_spark():
    global spark
    # Gjenbruk hvis 'spark' finnes og lever
    if 'spark' in globals():
        try:
            _ = spark.version
            print("Gjenbruker eksisterende SparkSession")
            return spark
        except Exception:
            pass  # faller gjennom og oppretter på nytt

    print("🚀 Oppretter ny SparkSession med Delta Sharing")
    spark = (
        SparkSession.builder
        .appName("DeltaSharingCDF")
        # .master("local[*]")  # bruk bare lokalt; ikke i Databricks
        .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )
    return spark

spark = ensure_spark()


Gjenbruker eksisterende SparkSession


# Hent endringer via CDF basert på tidspunkt

1. Beregner et starttidspunkt basert på antall timer tilbake i tid.  
2. Leser endringer fra tabellen siden dette tidspunktet.  
3. Hvis det finnes endringer, grupperes de etter endringstype (`insert`, `update`, `delete`).  
4. Viser en oppsummering av antall endringer per type.  
5. Viser inntil 10 eksempler for hver endringstype.


In [16]:
from pyspark.sql import SparkSession, functions as F
lookback_hours = 24
start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

cdf = (
    spark.read.format("deltaSharing")
    .option("responseFormat", "delta")
    .option("readChangeFeed", "true")
    .option("startingTimestamp", start_ts)
    .load(table_url)
)

if not cdf.rdd.isEmpty():
    change_summary = cdf.groupBy("_change_type").count().orderBy("_change_type")
    change_summary.show(truncate=False)

    for row in change_summary.collect():
        change_type = row["_change_type"]
        cdf.filter(F.col("_change_type") == change_type).show(10, truncate=False)


/var/folders/sh/t1kb_fwn67l_f8zrzb26bct40000gn/T/ipykernel_32310/2293493810.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


+----------------+-----+
|_change_type    |count|
+----------------+-----+
|insert          |461  |
|update_postimage|459  |
|update_preimage |459  |
+----------------+-----+

+-------------+------------+------------------------------------+----------------+--------------------+-----------+-----------------+--------------------------------------------------------------------------+-------------+---------+--------------------------+-------------------+------------+---------------+-----------------------+
|kulturminneId|enkeltminner|kulturminneId_date_updated          |lokalitetsnummer|matrikkelforingsdato|oppdatertAv|sistOppdatertDato|uuid                                                                      |versjon      |versjonId|from_datetime             |to_datetime        |_change_type|_commit_version|_commit_timestamp      |
+-------------+------------+------------------------------------+----------------+--------------------+-----------+-----------------+-------------------------

# Hent endringer basert på versjon

I stedet for å bruke tidsstempel, kan du også lese endringer mellom to spesifikke versjoner av tabellen.  
Dette er nyttig når du vet nøyaktig hvilke versjoner du vil sammenligne.

### 🔍 Finn tilgjengelige CDF-versjoner

Denne cellen finner **hvilke versjoner av tabellen som har Change Data Feed (CDF) tilgjengelig**.  
Det brukes for å vite hvilke versjoner man faktisk kan lese endringer fra med `readChangeFeed=true`.

Funksjonen `discover_cdf_versions(table_url)` returnerer:
- `earliest_cdf` og `latest_cdf`: første og siste versjon som kan leses via CDF  
- `versions_available`: alle CDF-versjoner som faktisk finnes  

Dette trinnet brukes for å finne riktig versjonsintervall før man henter endringsdata.  


In [17]:
info = cdf_utils.discover_cdf_versions(spark,table_url)


print("Earliest CDF:", info["earliest_cdf"])
print("Latest CDF  :", info["latest_cdf"])
print("Alle tilgjengelige CDF-versjoner:", info["versions_available"])


25/10/10 14:08:56 ERROR RetryUtils: Error during retry attempt 1, retryDuration=840, totalDuration=840 : HTTP request failed with status: HTTP/1.1 400 Bad Request {"error_code":"INVALID_PARAMETER_VALUE","message":"DS_UNSUPPORTED_DELTA_TABLE_FEATURES: Table features delta.enableDeletionVectors, delta.columnMapping.mode are found in table version: 257. historyShared:true, startVersion: 0. For DeletionVectors, use DBR with version 14.1(14.2 for CDF and streaming) or higher, or delta-sharing-spark with version 3.1 or higher, and set option (\"responseFormat\", \"delta\") to query the table. Or use delta_sharing python connector with version 1.1 or higher. Or ask your provider to disable DeletionVectors with\n (`ALTER TABLE <table_name> SET TBLPROPERTIES (delta.enableDeletionVectors=false)`),\n rewrite it without Deletion Vectors (`REORG TABLE <table_name> APPLY(PURGE)`).","details":[{"@type":"type.googleapis.com/google.rpc.ErrorInfo","reason":"DS_UNSUPPORTED_DELTA_TABLE_FEATURES","domain":

RuntimeError: Kunne ikke finne latest CDF-versjon: HTTP request failed with status: HTTP/1.1 400 Bad Request {"error_code":"INVALID_PARAMETER_VALUE","message":"DS_UNSUPPORTED_DELTA_TABLE_FEATURES: Table features delta.enableDeletionVectors, delta.columnMapping.mode are found in table version: 257. historyShared:true, startVersion: 0. For DeletionVectors, use DBR with version 14.1(14.2 for CDF and streaming) or higher, or delta-sharing-spark with version 3.1 or higher, and set option (\"responseFormat\", \"delta\") to query the table. Or use delta_sharing python connector with version 1.1 or higher. Or ask your provider to disable DeletionVectors with\n (`ALTER TABLE <table_name> SET TBLPROPERTIES (delta.enableDeletionVectors=false)`),\n rewrite it without Deletion Vectors (`REORG TABLE <table_name> APPLY(PURGE)`).","details":[{"@type":"type.googleapis.com/google.rpc.ErrorInfo","reason":"DS_UNSUPPORTED_DELTA_TABLE_FEATURES","domain":"data-sharing.databricks.com","metadata":{"historyShar

In [ ]:
version = 257
cdf = (spark.read.format("deltaSharing")
       .option("responseFormat","delta")
       .option("readChangeFeed","true")
       .option("startingVersion", version)
       .option("endingVersion", version)
       .load(table_url))
cdf.show()

+-------------+------------+--------------------------+----------------+--------------------+-----------+-----------------+--------------------+-------------+---------+--------------------+--------------------+----------------+---------------+--------------------+
|kulturminneId|enkeltminner|kulturminneId_date_updated|lokalitetsnummer|matrikkelforingsdato|oppdatertAv|sistOppdatertDato|                uuid|      versjon|versjonId|       from_datetime|         to_datetime|    _change_type|_commit_version|   _commit_timestamp|
+-------------+------------+--------------------------+----------------+--------------------+-----------+-----------------+--------------------+-------------+---------+--------------------+--------------------+----------------+---------------+--------------------+
|    323987400|        NULL|      323987400_2025-09...|           20570|          2010-01-15|     rastsv|       2025-09-18|{https://data.geo...|1758222926882|       58|2025-09-18 21:15:...| 9999-01-01 01:0

25/10/10 13:05:50 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 136241 ms exceeds timeout 120000 ms
25/10/10 13:05:50 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/10 13:05:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$